# Resonance Lattice — Fabric deploy

Deploys a `.rlat` knowledge model from `Files/rlat/<KM_NAME>.rlat` into the `passages_raw` table on an Eventhouse-backed KQL database. The materialised view `passages` (= `arg_max(ingest_time, *) by content_hash` over `passages_raw`) is the queryable surface the data agent's `rlat_search()` function reads from.

Re-run anytime to refresh: ingest appends to `passages_raw`; the view dedupes on `content_hash`, so the queryable row count stays at `n_distinct_content_hash` regardless of how many times you deploy the same corpus.

## Prerequisites

1. An Eventhouse + default KQL database in this workspace. Create one by running `scripts/fabric_e2e_provision_eventhouse.py` from a local checkout, or via the Fabric portal.
2. A `.rlat` file at `Files/rlat/<KM_NAME>.rlat`. Build one with the `fabric_build.ipynb` notebook in this folder.

The provisioning script prints `CLUSTER_URI`, `INGEST_URI`, and `KQL_DATABASE_NAME` — copy them into the parameter cell below.

## Session size

Default 2 vCores is fine for corpora up to ~10K passages. For larger ingests, uncomment the `%%configure` cell below and bump `vCores`. Session spinup is ~2 minutes regardless of vCore count — for a 100-passage smoke test the default is faster end-to-end.

In [ ]:
# %%configure -f
# {
#     "vCores": 8  // recommended: 4, 8, 16, 32, 64
# }


## Parameters

`KM_NAME` selects which `.rlat` to deploy: `Files/rlat/<KM_NAME>.rlat`. `KQL_DATABASE_NAME` is the default KQL DB created with the Eventhouse (same display name as the Eventhouse itself).

In [ ]:
KM_NAME = "team-docs"
KQL_DATABASE_NAME = "rlat-data-agent"

# From `fabric_e2e_provision_eventhouse.py` output.
CLUSTER_URI = "https://trd-xxxx.z5.kusto.fabric.microsoft.com"
INGEST_URI  = "https://ingest-trd-xxxx.z5.kusto.fabric.microsoft.com"


In [ ]:
%pip install -q "rlat>=2.1.0a15" "azure-kusto-data>=6.0" "azure-kusto-ingest>=6.0"


In [ ]:
from pathlib import Path
import notebookutils

LAKEHOUSE = notebookutils.lakehouse.getWithProperties(
    notebookutils.runtime.context["defaultLakehouseName"]
)

ONELAKE_RLAT = f"Files/rlat/{KM_NAME}.rlat"
LOCAL_RLAT   = Path(f"/tmp/{KM_NAME}.rlat")

abfss_rlat = f"{LAKEHOUSE['properties']['abfsPath']}/{ONELAKE_RLAT}"
if not notebookutils.fs.exists(abfss_rlat):
    raise FileNotFoundError(
        f"{ONELAKE_RLAT} not found in {LAKEHOUSE['displayName']!r}. "
        f"Build it first with fabric_build.ipynb."
    )

if LOCAL_RLAT.exists():
    LOCAL_RLAT.unlink()
notebookutils.fs.cp(abfss_rlat, f"file://{LOCAL_RLAT}", recurse=False)
print(f"downloaded {ONELAKE_RLAT} -> {LOCAL_RLAT}")


## Connect to the KQL database

Auth uses the notebook's own identity (managed by Fabric). The Kusto Python SDK accepts a token-provider callback — we wire it to `notebookutils.credentials.getToken` against the cluster URI.

In [ ]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.ingest import IngestionProperties, QueuedIngestClient, ReportLevel


def _kcsb(uri: str) -> KustoConnectionStringBuilder:
    # The azure-kusto SDK uses `with_token_provider(uri, callable)` —
    # `with_aad_token_provider_authentication` doesn't exist on this
    # release. notebookutils' "kusto" audience alias resolves to the
    # right scope for Fabric Eventhouse clusters.
    return KustoConnectionStringBuilder.with_token_provider(
        uri, lambda: notebookutils.credentials.getToken("kusto"),
    )


kc = KustoClient(_kcsb(CLUSTER_URI))
ic = QueuedIngestClient(_kcsb(INGEST_URI))
print(f"connected to {CLUSTER_URI} / {KQL_DATABASE_NAME}")


## Ensure schema

Creates `passages_raw` (append-only) + `Vector16` encoding on the `vector` column + sharding/merge policies tuned for an F2 single-node Eventhouse + the `passages` materialised view. Every statement is idempotent across re-runs.

In [ ]:
PRE_MV_DDL = [
    """
.create-merge table passages_raw (
    passage_id:string,
    layer:string,
    text:string,
    source_file:string,
    char_start:int,
    char_end:int,
    content_hash:string,
    vector:dynamic,
    confidence:string,
    cited_passage_ids:dynamic,
    created_utc:datetime,
    ingest_time:datetime
)
""".strip(),
    ".alter column passages_raw.vector policy encoding type='Vector16'",
    ".alter-merge table passages_raw policy sharding "
        "'{\"ShardEngineMaxRowCount\": 8000}'",
    ".alter-merge table passages_raw policy merge "
        "'{\"RowCountUpperBoundForMerge\": 8000}'",
]

# `with (backfill=true)` is only legal on `.create materialized-view`,
# not on `.create-or-alter` — re-running the alter raises
# MaterializedViewInvalidArgumentException. Guard with an existence
# probe; the definition is fixed in this notebook so no alter needed.
MV_CREATE = """
.create materialized-view with (backfill=true)
    passages on table passages_raw
{
    passages_raw
    | summarize arg_max(ingest_time, *) by content_hash
}
""".strip()

for stmt in PRE_MV_DDL:
    kc.execute_mgmt(KQL_DATABASE_NAME, stmt)
    print(f"  DDL ok: {stmt.splitlines()[0][:80]}")

mv_exists = kc.execute_mgmt(
    KQL_DATABASE_NAME,
    ".show materialized-views | where Name == 'passages' | count",
).primary_results[0].rows[0][0] > 0

if mv_exists:
    print("  DDL ok: passages materialized-view already exists (no-op)")
else:
    kc.execute_mgmt(KQL_DATABASE_NAME, MV_CREATE)
    print("  DDL ok: .create materialized-view passages (with backfill=true)")


## Read the knowledge model

Loads the base band (768-d gte-modernbert vectors) and the passage registry from the local `.rlat`. Bundled-mode `.rlat`s carry source text inside the ZIP; local-mode .rlats fetch text by re-opening the source file. Either works here — we only need the text bytes.

In [ ]:
import time
from datetime import datetime, timezone

import pandas as pd
from resonance_lattice.store import archive, open_store

t0 = time.time()
contents = archive.read(LOCAL_RLAT)
store    = open_store(LOCAL_RLAT, contents)
base     = contents.bands["base"]  # (N, 768) float32
assert base.shape[1] == 768, (
    f"P1 deploy expects 768-d base band (gte-modernbert); got {base.shape}. "
    f"Re-build the .rlat with the v2.x encoder."
)

now_iso = datetime.now(timezone.utc)
rows = []
for i, p in enumerate(contents.registry):
    text = store.fetch(p.source_file, p.char_offset, p.char_length)
    rows.append({
        "passage_id":        p.passage_id,
        "layer":             "source",
        "text":              text,
        "source_file":       p.source_file,
        "char_start":        int(p.char_offset),
        "char_end":          int(p.char_offset + p.char_length),
        "content_hash":      p.content_hash,
        "vector":            base[i].astype("float32").tolist(),
        "confidence":        "verified",
        "cited_passage_ids": [],
        "created_utc":       now_iso,
        "ingest_time":       now_iso,
    })
frame = pd.DataFrame(rows)
print(f"frame: {len(frame)} rows, build took {time.time()-t0:.1f}s")


## Ingest

Queues a single batch into `passages_raw`. The Eventhouse default batching window flushes within ~30 seconds; the visibility wait polls `passages_raw | count` until it reflects the new rows.

In [ ]:
import uuid

src_id = uuid.uuid4()
props = IngestionProperties(
    database=KQL_DATABASE_NAME,
    table="passages_raw",
    report_level=ReportLevel.FailuresAndSuccesses,
)
props.source_id = src_id

# Count before so we know when ingest has landed.
before = kc.execute(KQL_DATABASE_NAME, "passages_raw | count").primary_results[0].rows[0][0]
target = before + len(frame)

t0 = time.time()
ic.ingest_from_dataframe(frame, ingestion_properties=props)
print(f"queued ingest id={src_id}, waiting for passages_raw to reach {target}")

last = -1
deadline = time.time() + 300  # 5 min timeout
while time.time() < deadline:
    cnt = kc.execute(KQL_DATABASE_NAME, "passages_raw | count").primary_results[0].rows[0][0]
    if cnt != last:
        print(f"  passages_raw count = {cnt}")
        last = cnt
    if cnt >= target:
        break
    time.sleep(5)

after = last
elapsed = time.time() - t0
view_count = kc.execute(KQL_DATABASE_NAME, "passages | count").primary_results[0].rows[0][0]

import json
print()
print(json.dumps({
    "km_name":                   KM_NAME,
    "kql_database":              KQL_DATABASE_NAME,
    "rows_ingested":             len(frame),
    "passages_raw_before":       before,
    "passages_raw_after":        after,
    "passages_view_count":       view_count,
    "ingest_to_visible_seconds": round(elapsed, 1),
    "ingest_source_id":          str(src_id),
}, indent=2))


## What now

The materialised view `passages` is the queryable surface. The data agent's `rlat_search()` function (added in Phase 3 of the rlat × Fabric Data Agent integration) reads from this view; the agent calls the function via NL2KQL.

Schedule this notebook from a Fabric Pipeline if you want hands-off refresh — each run appends the current corpus; the view dedupes on `content_hash` automatically.